# Cervical Cancer Stage Classification on Kaggle

This notebook launches the backend training script with Kaggle-friendly paths. It looks for the repository, finds the dataset, and writes checkpoints to the Kaggle working directory.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_NAME = 'Cervical-Cancer-Classifier'


def find_backend_dir() -> Path | None:
    search_roots = [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd()]
    direct_candidates = [
        Path('/kaggle/working/backend'),
        Path('/kaggle/input/backend'),
        Path.cwd() / 'backend',
        Path('/kaggle/working') / REPO_NAME / 'backend',
        Path('/kaggle/input') / REPO_NAME / 'backend',
    ]

    for candidate in direct_candidates:
        if (candidate / 'train.py').exists():
            return candidate

    for root in search_roots:
        if not root.exists():
            continue
        for match in root.rglob('train.py'):
            if match.name == 'train.py' and match.parent.name == 'backend':
                return match.parent
    return None


def clone_repo_if_needed() -> Path:
    work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    repo_dir = work_root / REPO_NAME
    backend_dir = repo_dir / 'backend'
    if (backend_dir / 'train.py').exists():
        return backend_dir

    if repo_dir.exists():
        print(f'Removing incomplete repo folder: {repo_dir}')
        subprocess.run(['rm', '-rf', str(repo_dir)], check=False)

    print(f'Cloning repository from {REPO_URL}')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    if not (backend_dir / 'train.py').exists():
        raise FileNotFoundError(f'Cloned repo but could not find backend/train.py in {backend_dir}')
    return backend_dir


BACKEND_DIR = find_backend_dir()
if BACKEND_DIR is None:
    BACKEND_DIR = clone_repo_if_needed()

REPO_ROOT = BACKEND_DIR.parent
os.chdir(REPO_ROOT)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('Python:', sys.executable)

In [ ]:
def find_data_dir() -> Path | None:
    raw_candidates = [
        os.environ.get('DATA_DIR', ''),
        '/kaggle/input/datasets/shubhrawat132/herlevdataset',
        '/kaggle/input/Herlev Dataset',
        '/kaggle/input/herlev-dataset',
        '/kaggle/input/herlevdataset',
        '/kaggle/input/cervical-cancer-stage-classification',
        '/kaggle/input/cervical-cancer-dataset',
        str(REPO_ROOT / 'Herlev Dataset'),
        str(REPO_ROOT / 'data'),
    ]
    for candidate_text in raw_candidates:
        if not candidate_text:
            continue
        candidate = Path(candidate_text)
        if candidate.exists():
            return candidate
    return None


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working/Checkpoints') if Path('/kaggle/working').exists() else (REPO_ROOT / 'backend' / 'Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell.')

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
train_script = BACKEND_DIR / 'train.py'
command = [
    sys.executable,
    str(train_script),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '90',
    '--batch-size', '16',
    '--img-size', '224',
    '--backbone', 'tf_efficientnetv2_m',
    '--lr-head', '3e-4',
    '--lr-backbone', '3e-5',
    '--phase1-epochs', '24',
    '--phase2-epochs', '26',
    '--mixup-alpha', '0.30',
    '--focal-gamma', '1.5',
    '--patience', '15',
    '--num-workers', str(min(4, os.cpu_count() or 2)),
]

print('Running:')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
artifacts = sorted(OUTPUT_DIR.glob('*'))
print('Training artifacts:')
for artifact in artifacts:
    print('-', artifact.name)

metrics_path = OUTPUT_DIR / 'metrics.json'
print('metrics.json exists:', metrics_path.exists())
if metrics_path.exists():
    print('metrics.json saved at', metrics_path)